# EDA — corpus overview and descriptive statistics

Exploratory summaries of the annotated corpus and the cross-community corpus, reproducing the descriptive figures reported in the ERP (class balance, comments per channel, comment length, and RQ2 community composition). Runs from the included data — no collection or GPU required.

Reads `data/processed/` (labelled corpus and splits) and `outputs/rq2/` (scored corpus); writes figures to `outputs/eda/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
proc = PROJECT_ROOT / "data" / "processed"
figs = PROJECT_ROOT / "outputs" / "eda"
figs.mkdir(parents=True, exist_ok=True)

labelled = pd.read_csv(proc / "labelled_comments_final.csv")
label_cols = ["label_1A", "label_1B", "label_2"]
for c in label_cols:
    labelled[c] = pd.to_numeric(labelled[c], errors="coerce").astype(int)
print("Annotated corpus:", len(labelled), "comments")


## 1. Class balance of the annotated corpus

In [ ]:
n = len(labelled)
balance = pd.DataFrame({
    "positive_n": [int(labelled[c].sum()) for c in label_cols],
    "positive_pct": [round(100 * labelled[c].mean(), 1) for c in label_cols],
}, index=["1A grievance", "1B weaponised", "2 conspiratorial"])
print(balance)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(balance.index, balance["positive_pct"], color=["#3f6fb0", "#c0504d", "#4f8a5b"])
for i, v in enumerate(balance["positive_pct"]):
    ax.text(i, v + 0.5, f"{v}%", ha="center", fontsize=9)
ax.set_ylabel("Positive comments (%)")
ax.set_title(f"Class balance (n = {n})")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(figs / "class_balance.png", dpi=200); plt.show()


## 2. Comments per channel (annotated corpus)

In [ ]:
by_channel = labelled["channel_handle"].value_counts()
print(by_channel)
fig, ax = plt.subplots(figsize=(7, 4))
by_channel.sort_values().plot.barh(ax=ax, color="#6f93be")
ax.set_xlabel("Annotated comments"); ax.set_title("Comments per channel (annotated corpus)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(figs / "comments_per_channel.png", dpi=200); plt.show()


## 3. Comment-length distribution

In [ ]:
labelled["n_words"] = labelled["text_clean"].fillna("").str.split().str.len()
print(labelled["n_words"].describe().round(1))
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(labelled["n_words"], bins=40, color="#9db4d0", edgecolor="white")
ax.set_xlabel("Words per comment"); ax.set_ylabel("Comments")
ax.set_title("Comment-length distribution (annotated corpus)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(figs / "comment_length.png", dpi=200); plt.show()


## 4. Splits — sizes and positive counts

In [ ]:
rows = []
for name in ["train_fit", "validation", "test"]:
    d = pd.read_csv(proc / f"{name}.csv")
    for c in label_cols:
        d[c] = pd.to_numeric(d[c], errors="coerce").astype(int)
    rows.append({"split": name, "n": len(d),
                 "1A": int(d["label_1A"].sum()),
                 "1B": int(d["label_1B"].sum()),
                 "2":  int(d["label_2"].sum())})
print(pd.DataFrame(rows).to_string(index=False))


## 5. RQ2 cross-community corpus composition

In [ ]:
rq2 = pd.read_csv(PROJECT_ROOT / "outputs" / "rq2" / "rq2_FINAL_primary_12mo_SCORED.csv",
                  low_memory=False)
comp = rq2.groupby("community_group").agg(
    channels=("channel_name", "nunique"),
    videos=("video_id", "nunique"),
    comments=("comment_id", "count")).reset_index()
print(comp.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(comp["community_group"], comp["comments"], color="#3f6fb0")
ax.set_ylabel("Comments"); ax.set_title("RQ2 corpus size by community")
ax.tick_params(axis="x", rotation=20)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(figs / "rq2_community_size.png", dpi=200); plt.show()


## 6. RQ1 model comparison (baseline and transformer F1)

Positive-class F1 for each construct across the model ladder. Baseline scores (lexicon, Naïve Bayes, logistic regression) are read from the committed cross-validation results; the transformer bars show the selected-configuration **validation** F1 from notebooks 09-10.

In [ ]:
# RQ1 model comparison figure (baseline CV + transformer validation F1)
cvb = pd.read_csv(PROJECT_ROOT / "reference_results" / "cv_baseline_metrics.csv")

order = ["1A", "1B", "2"]
def baseline_row(model):
    s = cvb[cvb["model"] == model].set_index("construct")["f1_mean"]
    return [round(float(s[c]), 3) for c in order]

constructs = ["1A\ngrievance", "1B\nweaponised", "2\nconspiracy"]
results = {
    "Lexicon":             baseline_row("lexicon"),
    "Naive Bayes":         baseline_row("naive_bayes"),
    "Logistic regression": baseline_row("logreg"),
    "RoBERTa-base":        [0.661, 0.524, 0.655],   # validation F1 (notebooks 09-10)
    "DeBERTa-v3-base":     [0.659, 0.540, 0.685],   # validation F1 (notebooks 09-10)
}
colours = ["#c9d4e3", "#9db4d0", "#6f93be", "#3f6fb0", "#1f3864"]

x = np.arange(len(constructs))
n = len(results)
width = 0.8 / n

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (model, scores) in enumerate(results.items()):
    bars = ax.bar(x + (i - (n - 1) / 2) * width, scores, width,
                  label=model, color=colours[i % len(colours)],
                  edgecolor="white", linewidth=0.6)
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=7)

ax.set_ylabel("Positive-class F1")
ax.set_xticks(x, constructs)
ax.set_ylim(0, 0.8)
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linewidth=0.5, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
fig.savefig(figs / "rq1_model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", figs / "rq1_model_comparison.png")

## 7. RQ2 community prevalence (forest plot)

Model-detected prevalence of each construct across the four community types, with 95% bootstrap confidence intervals. Reads the committed `rq2_bootstrap_community_CI.csv`.

In [ ]:
# ============================================================
# RQ2 — COMMUNITY PREVALENCE FIGURE
# Three stacked panels using saved estimates and intervals.
# No analyses are rerun.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MultipleLocator

# ------------------------------------------------------------
# 1. LOAD SAVED RESULTS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rq2" / "rq2_prevalence"

INPUT_PATH = OUTPUT_DIR / "rq2_bootstrap_community_CI.csv"

results = pd.read_csv(
    INPUT_PATH,
    dtype={"construct": str, "community": str}
)

required = [
    "construct",
    "community",
    "prevalence_percent",
    "ci_low_percent",
    "ci_high_percent",
]

missing = set(required) - set(results.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

communities = [
    "manosphere_adjacent",
    "male_oriented_wellbeing",
    "mainstream_self_improvement",
    "unrelated_control",
]

labels = [
    "Manosphere-adjacent",
    "Male-oriented wellbeing",
    "Mainstream self-improvement",
    "Unrelated control",
]

# Construct, heading, axis maximum, tick spacing
panels = [
    ("1A", "Grievance/victimhood", 40, 10),
    ("1B", "Weaponised victimhood", 10, 2),
    ("2", "Conspiratorial framing", 10, 2),
]

if results.duplicated(["construct", "community"]).any():
    raise ValueError("Duplicate construct–community results found.")

results = results.set_index(["construct", "community"])

expected = pd.MultiIndex.from_product(
    [[panel[0] for panel in panels], communities],
    names=["construct", "community"]
)

if not expected.isin(results.index).all():
    raise ValueError("Expected construct–community results are missing.")

results = results.loc[expected].copy()

numeric_columns = [
    "prevalence_percent",
    "ci_low_percent",
    "ci_high_percent",
]

results[numeric_columns] = results[numeric_columns].apply(
    pd.to_numeric, errors="raise"
)

values = results[numeric_columns].to_numpy()

if not np.isfinite(values).all():
    raise ValueError("Missing or non-finite estimates found.")

if ((values < 0) | (values > 100)).any():
    raise ValueError("Expected percentages between 0 and 100.")

if (results["ci_low_percent"] > results["ci_high_percent"]).any():
    raise ValueError("A confidence interval has reversed bounds.")

# ------------------------------------------------------------
# 2. DRAW FIGURE
# ------------------------------------------------------------

NAVY = "#203B56"
BLUE = "#637F96"
TEXT = "#252525"
GRID = "#E6E9EC"

sns.set_theme(style="white", context="paper")

with plt.rc_context({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Arial", "Liberation Sans", "DejaVu Sans"
    ],
    "font.size": 10.5,
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "xtick.color": "#555555",
    "ytick.color": TEXT,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}):
    fig, axes = plt.subplots(
        3, 1,
        figsize=(7.5, 6.2),
        sharex=False
    )

    for ax, (construct, title, axis_max, tick_step) in zip(
        axes, panels
    ):
        data = results.xs(
            construct, level="construct"
        ).loc[communities]

        # Subtle shading for the manosphere cohort.
        ax.axhspan(
            -0.40, 0.40,
            color="#F0F4F7",
            zorder=0
        )

        for y, community in enumerate(communities):
            row = data.loc[community]

            estimate = float(row["prevalence_percent"])
            low = float(row["ci_low_percent"])
            high = float(row["ci_high_percent"])

            # Prevent silently clipping any confidence interval.
            if not (
                0 <= low <= high <= axis_max
                and 0 <= estimate <= axis_max
            ):
                raise ValueError(
                    f"{construct}, {community}: estimate or "
                    f"interval falls outside 0–{axis_max}%."
                )

            colour = NAVY if y == 0 else BLUE

            # Confidence interval
            ax.plot(
                [low, high],
                [y, y],
                color=colour,
                linewidth=2,
                solid_capstyle="round",
                zorder=2
            )

            # Point estimate
            ax.scatter(
                estimate, y,
                s=38,
                color=colour,
                edgecolor="white",
                linewidth=0.6,
                zorder=3,
                clip_on=False
            )

            # Percentages aligned in a separate column.
            ax.text(
                1.055, y,
                f"{estimate:.2f}%",
                transform=ax.get_yaxis_transform(),
                ha="left",
                va="center",
                fontsize=11,
                fontweight="bold" if y == 0 else "normal",
                color=colour,
                clip_on=False
            )

        ax.text(
            -0.64, 1.15,
            f"{construct}   {title}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=12,
            fontweight="bold"
        )

        ax.set_yticks(range(4))
        ax.set_yticklabels(labels, fontsize=10.5)
        ax.set_ylim(3.50, -0.55)

        # Different scales, clearly labelled in each panel.
        ax.set_xlim(0, axis_max)
        ax.xaxis.set_major_locator(MultipleLocator(tick_step))

        ax.tick_params(
            axis="x",
            labelsize=9.5,
            length=0,
            pad=5
        )
        ax.tick_params(axis="y", length=0, pad=12)

        ax.set_axisbelow(True)
        ax.grid(axis="x", color=GRID, linewidth=0.7)
        ax.grid(axis="y", visible=False)

        for spine in ax.spines.values():
            spine.set_visible(False)

    axes[-1].set_xlabel(
        "Model-detected prevalence (%)",
        fontsize=11,
        labelpad=10
    )

    fig.subplots_adjust(
        left=0.39,
        right=0.88,
        top=0.92,
        bottom=0.10,
        hspace=0.68
    )

    # --------------------------------------------------------
    # 3. SAVE PNG AND PDF
    # --------------------------------------------------------

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for extension in ["png", "pdf"]:
        path = OUTPUT_DIR / (
            f"rq2_prevalence_publication.{extension}"
        )
        fig.savefig(
            path,
            dpi=400,
            facecolor="white"
        )
        print("Saved:", path)

    plt.show()
